# Experimentul 11: optimizarea observabilei de readout pentru STAR ZZ

**Pornim exclusiv de la Exp. 9, topologia STAR** (4 qubiți, ZZ ×1 pe perechile (0,1), (0,2), (0,3), ansatz EfficientSU2 ×3, 32 de parametri, 15 CNOT, 73 locuri de zgomot, readout ⟨Z₀⟩). **Singura modificare: observabila măsurată.** Scorul de fraudă rămâne s(x) = (1 − ⟨O⟩) / 2, fără straturi clasice; clasificarea vine din circuit.

| Model | Observabilă O | Antrenare |
|---|---|---|
| **A** (baseline) | Z₀ | ponderi Exp. 9 `weights/star/` (verificate, nereantrenate) |
| **B** | Z₁ | de la zero, 5 seeds |
| **C** | Z₂ | de la zero, 5 seeds |
| **D** | Z₃ | de la zero, 5 seeds |
| **E** | Z₀Z₁ (paritate) | de la zero, 5 seeds |

## Întrebarea și ipotezele (de testat, nu presupuse)
- **Întrebare:** schimbând doar O, putem crește recall-ul fraudelor la pragul oficial max-F1 păstrând AUPRC la p = 0.05 aproape de Z₀ (toleranță internă 0.02)?
- **H1:** un qubit periferic (1, 2, 3) sau paritatea Z₀Z₁ citesc o combinație a informației mai potrivită pentru fraudă.
- **H0:** Z₀ rămâne cel puțin la fel de bun; diferențele sunt în zgomotul dintre seeds.
- **Separat:** robustețea (AUPRC(0) − AUPRC(0.05), Spearman ρ) nu este același lucru cu performanța de bază (AUPRC la p = 0).

## Protocol (identic cu Exp. 9)
Split `split_indices.npz`, `undersample(X_train, y_train, seed)` 1:1 (590), `AngleEncoder(4, seed)` (StandardScaler pe Time/Amount, PCA(4) fitat pe setul 1:1, MinMaxScaler → [0, π]), `init_weights(seed)` și ordinea mini-batch-urilor `torch.Generator().manual_seed(seed)` identice pentru toate observabilele, Adam lr 0.05, BCE, batch 32, 30 de epoci, antrenare fără zgomot, checkpoint după AUPRC maximă pe validation (p = 0). Evaluare cu `noisy_qnode` oficial (decompoziție în `NATIVE_GATES`, `DepolarizingChannel(p)` după fiecare poartă pe fiecare qubit, `default.mixed`), p = 0, 0.001, 0.01, 0.05, prag max-F1 pe validation la același p.

**Doar TRAIN și VALIDATION.** Testul nu se folosește. Fără Exp. 10, fără rotation fusion, fără ZNE / noise-aware training, encoder și circuit neschimbate. Codul comun (copiat neschimbat din Exp. 9 + observabila ca argument + funcțiile pentru conul cauzal) este în `src/quantum_readout.py`.

## 1. Configurație și imports

**Seeds.** `set_seed` fixează numpy și torch; amestecarea folosește un `torch.Generator` dedicat (ca în Exp. 3 / 9); punctele aleatoare pentru verificări vin din `np.random.default_rng(42)`. Simulatoarele PennyLane rulează cu `shots=None` (valori exacte); excepția este numărarea execuțiilor parameter-shift cu `qml.Tracker`, pe un dispozitiv cu 1024 shots și `seed` fix. Dtype implicit torch: float64 (ca în Exp. 9).

In [ ]:
import hashlib
import json
import math
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import train_test_split

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))
from src.quantum_readout import (ANSATZ_REPS, BATCH_SIZE, EPOCHS, FEATURE_MAP_REPS, LEARNING_RATE, N_QUBITS, NATIVE_GATES,
                                 OBSERVABLE_WIRES, OBSERVABLES, STAR_PAIRS, AngleEncoder, ansatz, backward_light_cone,
                                 choose_threshold, evaluate, init_weights, make_feature_map, make_model, make_vqc,
                                 noise_locations, noisy_predict, noisy_qnode, predict, readout, selective_noise_qnode,
                                 set_seed, train_vqc_run, undersample)
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# ---- singura modificare față de Exp. 9 (STAR): observabila ----
OBS = list(OBSERVABLES)                   # ["Z0", "Z1", "Z2", "Z3", "Z0Z1"]
BASELINE = "Z0"                           # Exp. 9 STAR, ponderi refolosite
NEW_OBS = [o for o in OBS if o != BASELINE]

# ---- fix, identic cu Exp. 9 ----
SPLIT_SEED = 42
SEEDS = [0, 1, 2, 3, 4]
NOISE_LEVELS = [0.001, 0.01, 0.05]
P_ALL = [0.0, *NOISE_LEVELS]
SHOTS_PER_EXECUTION = 1024    # doar pentru cost ledger
N_BALANCED = 590

# ---- analiză ----
RECALL_TARGETS = [0.70, 0.75, 0.80, 0.85]   # analiză secundară, exploratorie
AUPRC_TOLERANCE = 0.02                      # criteriu intern de proiect, nu regulă oficială
P_FOCUS = 0.05
N_CONE_POINTS = 30                          # puncte (x, w) aleatoare pentru suportul gradientului
N_NOISE_CONE_POINTS = 5                     # puncte pentru verificarea „zgomotul din afara conului nu contează”

# ---- verificări ----
TORCH_DEFAULT_DTYPE = torch.float64
EXACT_TOL = 1e-12
GRAD_ZERO_TOL = 1e-10
SMOKE_ROWS = 2000
SMOKE_EPOCHS = 2

EXPERIMENT_NAME = "quantum_readout_optimization"
RESULTS_DIR = REPO_ROOT / "results" / "result_exp11"
WEIGHTS_DIR = RESULTS_DIR / "weights"
SCORES_DIR = RESULTS_DIR / "scores"
PLOTS_DIR = RESULTS_DIR / "plots"
EXP9_DIR = REPO_ROOT / "results" / "result_exp9"   # doar citit: ponderi STAR, unghiuri, istoric, metrici
EXP4_DIR = REPO_ROOT / "results" / "result_exp4"   # doar citit: regresia logistică pe aceleași unghiuri
SPLIT_PATH = REPO_ROOT / "split_indices.npz"

OBS_LABELS = {"Z0": "⟨Z₀⟩ (Exp. 9)", "Z1": "⟨Z₁⟩", "Z2": "⟨Z₂⟩", "Z3": "⟨Z₃⟩", "Z0Z1": "⟨Z₀Z₁⟩"}
OBS_COLORS = {"Z0": "#2a78d6", "Z1": "#eb6834", "Z2": "#1baf7a", "Z3": "#eda100", "Z0Z1": "#e87ba4"}   # paleta validată, sloturile 1-5
OBS_MARKERS = {"Z0": "o", "Z1": "s", "Z2": "D", "Z3": "^", "Z0Z1": "P"}
SEED_DOT_COLOR, LR_COLOR = "#8a8f98", "#52514e"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})

set_seed(SPLIT_SEED)
torch.set_default_dtype(TORCH_DEFAULT_DTYPE)
print(f"observabile: {OBS}   seeds: {SEEDS}   p: {P_ALL}   dtype implicit torch: {torch.get_default_dtype()}")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__)
print("rezultate:", RESULTS_DIR.relative_to(REPO_ROOT))

## 2. Datele și splitul comun

Identic cu Exp. 9: `split_indices.npz`, verificat față de reconstrucția oficială, plus hash-ul SHA-256 al indicilor (salvat în `config.json`). Din test păstrăm doar indicii; nici features, nici etichete.

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
for path in [DATA_PATH, SPLIT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and "Time" in df.columns and df.index.equals(pd.RangeIndex(len(df)))

X_all, y_all = df.drop(columns="Class"), df["Class"].to_numpy()
split = np.load(SPLIT_PATH)
train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]
SPLIT_SHA256 = hashlib.sha256(b"".join(np.ascontiguousarray(split[k]).tobytes() for k in ["train", "val", "test"])).hexdigest()

_trval, _test = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y_all, random_state=SPLIT_SEED)
_train, _val = train_test_split(_trval, test_size=0.25, stratify=y_all[_trval], random_state=SPLIT_SEED)
assert np.array_equal(_train, train_idx) and np.array_equal(_val, val_idx) and np.array_equal(_test, test_idx), "split_indices.npz diferă de splitul oficial"
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx))
assert (len(test_idx), int(y_all[test_idx].sum())) == (56962, 98)

X_train, y_train = X_all.loc[train_idx], y_all[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all[val_idx]
assert (len(train_idx), int(y_train.sum())) == (170883, 295)
assert (len(val_idx), int(y_val.sum())) == (56962, 99)
del X_all, y_all, df, _trval, _test, _train, _val, test_idx   # testul nu se folosește în Exp. 11

print(f"train {len(train_idx):,} ({int(y_train.sum())} fraude)   validation {len(val_idx):,} ({int(y_val.sum())} fraude)   test: neatins")
print("SHA-256 split (train + val + test):", SPLIT_SHA256)

## 3. Cele cinci circuite și verificarea observabilelor

`CIRCUITS[o] = make_vqc(STAR_PAIRS, o)`: feature map-ul STAR și ansatz-ul din Exp. 9, cu `qml.expval(OBSERVABLES[o]())`. Verificăm:

1. **Circuitul STAR = Exp. 9:** secvența de porți native este identică cu `native_op_order` salvat în `result_exp9/config.json`.
2. **Singura diferență este observabila:** pe 20 de intrări aleatoare, tape-urile celor cinci modele au exact aceleași operații (`qml.equal`, aceleași unghiuri); diferă doar măsurarea.
3. **Structura:** 4 qubiți, perechile ZZ (0,1), (0,2), (0,3), 15 CNOT, 73 locuri de zgomot, 32 de parametri, pentru fiecare observabilă. Observabilele Z sunt diagonale în baza computațională: nicio poartă de rotire a bazei nu se adaugă.
4. **Observabilele pe circuite cu rezultat analitic cunoscut:** stări de bază (±1), produs de RY (cos θ), starea Bell și amestecul clasic ½(|00⟩⟨00| + |11⟩⟨11|) (corelație de paritate fără entanglement), ⟨O⟩ din probabilitățile rezultatelor pe circuitul STAR real, cu și fără zgomot.
5. **Scorurile sunt în [0, 1]** și canalul depolarizant (check 10.1 din Exp. 9).

In [ ]:
CIRCUITS = {o: make_vqc(STAR_PAIRS, o) for o in OBS}
FEATURE_MAP = make_feature_map(STAR_PAIRS)
dev4 = qml.device("default.qubit", wires=N_QUBITS)   # shots=None: valori de așteptare exacte
QNODES = {o: qml.QNode(fn, dev4, interface="torch", diff_method="backprop") for o, fn in CIRCUITS.items()}
MODELS = {o: make_model(q) for o, q in QNODES.items()}   # fraud_probability = (1 - <O>) / 2


def same_ops(a, b):
    return len(a) == len(b) and all(qml.equal(o1, o2) for o1, o2 in zip(a, b))


def op_sequence(ops):
    return [f"{op.name}{list(op.wires)}" for op in ops]


def tape_of(circuit_fn, x, weights):
    qnode = qml.transforms.decompose(qml.QNode(circuit_fn, dev4), gate_set=NATIVE_GATES)
    return qml.workflow.construct_tape(qnode)(x, weights)


checks = []
exp9_config = json.loads((EXP9_DIR / "config.json").read_text(encoding="utf-8"))
assert [tuple(p) for p in exp9_config["topologies"]["star"]["pairs"]] == STAR_PAIRS

rng_check = np.random.default_rng(SPLIT_SEED)
X_check = rng_check.uniform(0, np.pi, (20, N_QUBITS))
W_check = rng_check.uniform(0, 2 * np.pi, (20, ANSATZ_REPS + 1, N_QUBITS, 2))
for x, w in zip(X_check, W_check):
    tapes = {o: tape_of(CIRCUITS[o], x, w) for o in OBS}
    assert op_sequence(tapes[BASELINE].operations) == exp9_config["topologies"]["star"]["native_op_order"], "STAR ≠ Exp. 9"
    for o in OBS:
        assert same_ops(tapes[o].operations, tapes[BASELINE].operations), f"{o}: alte porți decât STAR + Z0"
        assert len(tapes[o].measurements) == 1 and qml.equal(tapes[o].measurements[0].obs, OBSERVABLES[o]())
checks.append({"verificare": "STAR = Exp. 9 (ordinea porților native din config.json)", "intrări": len(X_check), "trecut": True})
checks.append({"verificare": "cele 5 modele au aceleași porți; diferă doar observabila măsurată", "intrări": len(X_check), "trecut": True})

x_probe, w_probe = X_check[0], init_weights(SEEDS[0])
structure_rows, circuit_ops = [], {}
for o in OBS:
    ops = tape_of(CIRCUITS[o], x_probe, w_probe.detach().numpy()).operations
    assert all(op.name in NATIVE_GATES for op in ops)
    resources = qml.specs(qml.transforms.decompose(qml.QNode(CIRCUITS[o], dev4), gate_set=NATIVE_GATES))(torch.tensor(x_probe), w_probe)["resources"]
    row = {"observabilă": o, "qubiți măsurați": str(OBSERVABLE_WIRES[o]), "qubiți": N_QUBITS,
           "perechi ZZ": " ".join(f"({i},{j})" for i, j in STAR_PAIRS), "porți": len(ops),
           "CNOT": sum(op.name == "CNOT" for op in ops), "porți 1 qubit": sum(len(op.wires) == 1 for op in ops),
           "porți 2 qubiți": sum(len(op.wires) == 2 for op in ops), "adâncime": resources.depth,
           "locuri de zgomot": noise_locations(CIRCUITS[o], torch.tensor(x_probe), w_probe),
           "parametri antrenabili (nominal)": w_probe.numel(), "porți de rotire a bazei": 0}
    assert (row["CNOT"], row["locuri de zgomot"], row["parametri antrenabili (nominal)"]) == (15, 73, 32), row
    assert row["locuri de zgomot"] == row["porți 1 qubit"] + 2 * row["porți 2 qubiți"]
    structure_rows.append(row)
    circuit_ops[o] = ops
STRUCT = pd.DataFrame(structure_rows).set_index("observabilă")
checks.append({"verificare": "4 qubiți, perechi (0,1)(0,2)(0,3), 15 CNOT, 73 locuri de zgomot, 32 parametri (toate observabilele)", "intrări": len(OBS), "trecut": True})
display(STRUCT)

In [ ]:
# 4. Observabilele pe circuite cu rezultat analitic cunoscut
PARITY = {"Z0": lambda b: b[0], "Z1": lambda b: b[1], "Z2": lambda b: b[2], "Z3": lambda b: b[3], "Z0Z1": lambda b: b[0] ^ b[1]}


@qml.qnode(dev4)
def basis_expvals(bits):
    qml.BasisState(np.array(bits), wires=range(N_QUBITS))
    return [qml.expval(OBSERVABLES[o]()) for o in OBS]


for k in range(2 ** N_QUBITS):   # (a) stări de bază: <O> = (-1)^(paritatea biților măsurați)
    bits = [(k >> (N_QUBITS - 1 - q)) & 1 for q in range(N_QUBITS)]
    assert np.allclose(basis_expvals(bits), [(-1) ** PARITY[o](bits) for o in OBS], atol=EXACT_TOL)


@qml.qnode(dev4)
def product_ry(thetas):
    for q in range(N_QUBITS):
        qml.RY(thetas[q], wires=q)
    return [qml.expval(OBSERVABLES[o]()) for o in OBS]


for thetas in rng_check.uniform(0, np.pi, (10, N_QUBITS)):   # (b) <Z_k> = cos θ_k, <Z0 Z1> = cos θ0 cos θ1
    c = np.cos(thetas)
    assert np.allclose(product_ry(thetas), [c[0], c[1], c[2], c[3], c[0] * c[1]], atol=EXACT_TOL)

dev2_mixed = qml.device("default.mixed", wires=2)


@qml.qnode(dev2_mixed)
def bell_like(dephase):
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    if dephase:
        qml.PhaseDamping(1.0, wires=0)   # elimină coerențele: amestec clasic ½|00><00| + ½|11><11|
    return qml.expval(qml.PauliZ(0)), qml.expval(qml.PauliZ(1)), qml.expval(qml.PauliZ(0) @ qml.PauliZ(1)), qml.density_matrix(wires=[0, 1])


def min_eig_partial_transpose(rho):
    rho = np.asarray(rho).reshape(2, 2, 2, 2)
    return float(np.linalg.eigvalsh(rho.transpose(0, 3, 2, 1).reshape(4, 4)).min())   # transpusa parțială pe qubitul 1


entanglement_demo = []
for name, dephase in [("Bell (|00⟩ + |11⟩)/√2", False), ("amestec clasic ½|00⟩⟨00| + ½|11⟩⟨11|", True)]:
    z0, z1, z0z1, rho = bell_like(dephase)
    neg = min_eig_partial_transpose(rho)
    entanglement_demo.append({"stare": name, "⟨Z₀⟩": float(z0), "⟨Z₁⟩": float(z1), "⟨Z₀Z₁⟩": float(z0z1),
                              "min. valoare proprie ρ^T₁ (< 0 ⇔ entanglement)": neg})
    assert abs(z0) < 1e-6 and abs(z1) < 1e-6 and abs(z0z1 - 1) < 1e-6
# PhaseDamping(1.0) lasă o coerență reziduală ~5e-8 (≈ √ε din sqrt(1 − γ)): toleranță 1e-6 pentru amestec
assert entanglement_demo[0]["min. valoare proprie ρ^T₁ (< 0 ⇔ entanglement)"] < -0.49 and entanglement_demo[1]["min. valoare proprie ρ^T₁ (< 0 ⇔ entanglement)"] > -1e-6
display(pd.DataFrame(entanglement_demo).set_index("stare"))


def probs_circuit(p):
    # circuitul STAR complet, cu probabilitățile celor 16 rezultate în loc de <O>
    def fn(x, weights):
        FEATURE_MAP(x)
        ansatz(weights)
        return qml.probs(wires=range(N_QUBITS))
    return noisy_qnode(fn, p)


BITS = np.array([[(k >> (N_QUBITS - 1 - q)) & 1 for q in range(N_QUBITS)] for k in range(2 ** N_QUBITS)])
SIGNS = {o: np.array([(-1) ** PARITY[o](b) for b in BITS]) for o in OBS}   # <O> = Σ_b (-1)^paritate(b) P(b)
max_probs_err = 0.0
for p in [0.0, 0.05]:
    for x, w in zip(X_check[:5], W_check[:5]):
        P = probs_circuit(p)(torch.tensor(x), torch.tensor(w)).numpy()
        for o in OBS:
            direct = float(noisy_qnode(CIRCUITS[o], p)(torch.tensor(x), torch.tensor(w)))
            max_probs_err = max(max_probs_err, abs(direct - SIGNS[o] @ P))
assert max_probs_err < 1e-12, max_probs_err
checks += [{"verificare": "observabile pe stări de bază (16) și produse de RY (10): valori analitice", "intrări": 26, "trecut": True},
           {"verificare": "Bell vs amestec clasic: ⟨Z₀Z₁⟩ = 1 în ambele, entanglement doar în Bell (PPT)", "intrări": 2, "trecut": True},
           {"verificare": "⟨O⟩ = Σ_b (−1)^paritate(b) P(b) pe circuitul STAR real, p = 0 și 0.05", "intrări": 10, "trecut": True, "eroare max": max_probs_err}]

# 5. scoruri în [0, 1] și canalul depolarizant (10.1)
score_probe = {o: predict(X_check, torch.tensor(W_check[0]), MODELS[o]) for o in OBS}
assert all((s >= -EXACT_TOL).all() and (s <= 1 + EXACT_TOL).all() for s in score_probe.values())


@qml.qnode(qml.device("default.mixed", wires=1))
def noisy_ry(theta, p):
    qml.RY(theta, wires=0)
    qml.DepolarizingChannel(p, wires=0)
    return qml.expval(qml.PauliZ(0))


for theta, p in [(0.3, 0.0), (1.0, 0.1), (2.5, 0.3), (0.7, 0.75)]:
    assert np.isclose(noisy_ry(theta, p), (1 - 4 * p / 3) * np.cos(theta))
checks.append({"verificare": "10.1 canal depolarizant: ⟨Z⟩ = (1 − 4p/3) cos θ", "intrări": 4, "trecut": True})
print(f"✅ observabile verificate analitic; ⟨O⟩ din probabilități: eroare max {max_probs_err:.1e}")

## 4. Conul cauzal invers al fiecărei observabile

Pentru o observabilă O pe qubiții M, în imaginea Heisenberg ⟨O⟩ = Tr(ρ₀ U†OU). O poartă poate schimba ⟨O⟩ doar dacă atinge un qubit care, în acel moment (mergând înapoi de la măsurare), este în suportul lui O evoluat. Calculăm trei niveluri, de la cel mai larg la cel mai strict:

1. **Conul pe fire** (`backward_light_cone`): mergem înapoi pe tape-ul decompus; o poartă care atinge mulțimea S de fire este în con și firele ei intră în S. Dă porțile, parametrii și **locurile de zgomot din con**: un canal depolarizant pe un fir care nu e în S acționează pe identitate în imaginea Heisenberg (orice canal CPTP are adjunctul unital), deci nu poate schimba ⟨O⟩.
2. **Parametrii cu gradient nenul** (exact, backprop) pe 30 de puncte (x, w) aleatoare: mai strict decât conul pe fire, pentru că structura Pauli contează (ex. RZ pe qubitul măsurat, ultima poartă, comută cu Z).
3. **Verificare numerică a conului de zgomot:** pe 5 puncte aleatoare, p = 0.05, zgomot doar în con = zgomot peste tot, iar zgomot doar în afara conului = fără zgomot.

**Atenție:** zgomotul se aplică în continuare **întregului circuit** în toate evaluările; conul spune doar care canale pot avea efect asupra observabilei.

Numărăm și execuțiile parameter-shift măsurate cu `qml.Tracker`, exact ca în Exp. 9 (mini-batch de 32 din setul 1:1 al seed-ului 42, `init_weights(0)`, 1024 shots).

In [ ]:
N_FM_OPS = 2 * N_QUBITS + 3 * len(STAR_PAIRS)   # H, RZ per qubit + CNOT, RZ, CNOT per pereche
PARAM_NAMES = [f"θ[{l},{q},{'RY' if g == 0 else 'RZ'}]" for l in range(ANSATZ_REPS + 1) for q in range(N_QUBITS) for g in range(2)]
an_ops = circuit_ops[BASELINE][N_FM_OPS:]
ROT_OP_INDEX = [N_FM_OPS + k for k, op in enumerate(an_ops) if op.name in ("RY", "RZ")]   # op index for each of the 32 params, in weights order
assert len(ROT_OP_INDEX) == 32 and [circuit_ops[BASELINE][k].name for k in ROT_OP_INDEX] == ["RY", "RZ"] * 16

rng_cone = np.random.default_rng(SPLIT_SEED)
cone_x = rng_cone.uniform(0, np.pi, (N_CONE_POINTS, N_QUBITS))
cone_w = rng_cone.uniform(0, 2 * np.pi, (N_CONE_POINTS, ANSATZ_REPS + 1, N_QUBITS, 2))

X_bal_check, y_bal_check = undersample(X_train, y_train, SPLIT_SEED)
Z_check = AngleEncoder(N_QUBITS, SPLIT_SEED).fit(X_bal_check).transform(X_bal_check)
batch_x, batch_y = torch.tensor(Z_check[:BATCH_SIZE]), torch.tensor(y_bal_check[:BATCH_SIZE], dtype=torch.float64)

cone_rows, cone_params, cone_masks = [], {}, {}
for o in OBS:
    ops = circuit_ops[o]
    in_cone, noisy_wires = backward_light_cone(ops, OBSERVABLE_WIRES[o])
    params_wire = np.array([in_cone[k] for k in ROT_OP_INDEX])

    # gradient exact pe 30 de puncte (parametri și unghiuri de intrare)
    grad_w, grad_x = torch.zeros(32), torch.zeros(N_QUBITS)
    for x, w in zip(cone_x, cone_w):
        xt, wt = torch.tensor(x, requires_grad=True), torch.tensor(w, requires_grad=True)
        QNODES[o](xt, wt).backward()
        grad_w, grad_x = torch.maximum(grad_w, wt.grad.abs().flatten()), torch.maximum(grad_x, xt.grad.abs())
    params_grad = (grad_w > GRAD_ZERO_TOL).numpy()
    assert not (params_grad & ~params_wire).any(), f"{o}: gradient nenul în afara conului pe fire"

    # parametrii cu gradient zero chiar nu schimbă <O> (perturbare mare)
    max_dead_effect = 0.0
    for x, w in zip(cone_x[:5], cone_w[:5]):
        w2 = w.copy().reshape(-1)
        w2[~params_grad] += 1.234
        max_dead_effect = max(max_dead_effect, abs(float(QNODES[o](torch.tensor(x), torch.tensor(w))) - float(QNODES[o](torch.tensor(x), torch.tensor(w2.reshape(w.shape))))))
    assert max_dead_effect < 1e-12, (o, max_dead_effect)

    # zgomotul din afara conului nu are efect; selective_noise_qnode cu toate canalele = noisy_qnode oficial
    all_wires = [list(op.wires) for op in ops]
    outside = [[v for v in a if v not in b] for a, b in zip(all_wires, noisy_wires)]
    err_official = err_cone = err_outside = 0.0
    for x, w in zip(cone_x[:N_NOISE_CONE_POINTS], cone_w[:N_NOISE_CONE_POINTS]):
        ops_xw = tape_of(CIRCUITS[o], x, w).operations
        full = float(selective_noise_qnode(ops_xw, all_wires, P_FOCUS, OBSERVABLES[o])())
        official = float(noisy_qnode(CIRCUITS[o], P_FOCUS)(torch.tensor(x), torch.tensor(w)))
        cone_only = float(selective_noise_qnode(ops_xw, noisy_wires, P_FOCUS, OBSERVABLES[o])())
        out_only = float(selective_noise_qnode(ops_xw, outside, P_FOCUS, OBSERVABLES[o])())
        clean = float(QNODES[o](torch.tensor(x), torch.tensor(w)))
        err_official, err_cone, err_outside = max(err_official, abs(full - official)), max(err_cone, abs(cone_only - full)), max(err_outside, abs(out_only - clean))
    assert max(err_official, err_cone, err_outside) < 1e-12, (o, err_official, err_cone, err_outside)

    # execuții parameter shift măsurate (ca în Exp. 9)
    shot_device = qml.device("default.qubit", wires=N_QUBITS, seed=SPLIT_SEED)
    hardware_like = qml.set_shots(qml.QNode(CIRCUITS[o], shot_device, interface="torch", diff_method="parameter-shift"), shots=SHOTS_PER_EXECUTION)
    w_tracker = init_weights(SEEDS[0])
    with qml.Tracker(shot_device) as tracker:
        loss = torch.nn.BCELoss()(readout(hardware_like(batch_x, w_tracker)), batch_y)
        loss.backward()
    per_example = tracker.totals["executions"] / len(batch_x)

    layer_wire = [int(params_wire[8 * l:8 * (l + 1)].sum()) for l in range(ANSATZ_REPS + 1)]
    layer_grad = [int(params_grad[8 * l:8 * (l + 1)].sum()) for l in range(ANSATZ_REPS + 1)]
    fm_in_cone = sum(in_cone[:N_FM_OPS])
    cone_rows.append({
        "observabilă": o, "qubiți măsurați": str(OBSERVABLE_WIRES[o]),
        "porți în con (pe fire)": int(sum(in_cone)), "porți total": len(ops),
        "porți feature map în con": int(fm_in_cone), "CNOT în con": int(sum(c and op.name == "CNOT" for c, op in zip(in_cone, ops))),
        "locuri de zgomot în con": int(sum(map(len, noisy_wires))), "locuri de zgomot total": int(STRUCT.loc[o, "locuri de zgomot"]),
        "parametri în conul pe fire": int(params_wire.sum()), "parametri cu gradient nenul (exact)": int(params_grad.sum()),
        "parametri nominali": 32,
        "parametri activi per strat ansatz (con pe fire)": str(layer_wire), "parametri activi per strat ansatz (gradient)": str(layer_grad),
        "unghiuri de intrare cu gradient nenul": int((grad_x > GRAD_ZERO_TOL).sum()),
        "execuții / exemplu (Tracker, parameter shift)": per_example,
        "parametri deplasați de parameter shift (Tracker)": (per_example - 1) / 2,
        "execuții / exemplu (nominal 1 + 2P)": 1 + 2 * 32,
        "execuții / exemplu dacă se deplasează doar parametrii cu gradient nenul (potențial, neimplementat)": 1 + 2 * int(params_grad.sum()),
        "max |Δ⟨O⟩| la perturbarea parametrilor cu gradient 0": max_dead_effect,
        "max |zgomot doar în con − zgomot peste tot|": err_cone, "max |zgomot doar în afara conului − fără zgomot|": err_outside,
        "max |selectiv (toate canalele) − noisy_qnode oficial|": err_official,
    })
    cone_params[o] = {"con pe fire": params_wire, "gradient nenul": params_grad}
    cone_masks[o] = {"in_cone": in_cone, "noisy_wires": noisy_wires}

CONE = pd.DataFrame(cone_rows).set_index("observabilă")
assert CONE.loc[BASELINE, "execuții / exemplu (Tracker, parameter shift)"] == pd.read_csv(EXP9_DIR / "circuit_comparison.csv", index_col=0).loc["star", "execuții / exemplu (Tracker, parameter shift)"]
assert (CONE["parametri deplasați de parameter shift (Tracker)"] == CONE["parametri în conul pe fire"]).all(), "Tracker ≠ conul pe fire"
checks += [{"verificare": f"{o}: gradient 0 ⇒ ⟨O⟩ neschimbat; zgomot în afara conului fără efect; zgomot doar în con = zgomot peste tot",
            "intrări": N_NOISE_CONE_POINTS, "trecut": True, "eroare max": max(CONE.loc[o, c] for c in CONE.columns if c.startswith("max |"))} for o in OBS]
checks.append({"verificare": "STAR + Z0: execuții Tracker = Exp. 9 (41); Tracker deplasează exact parametrii din conul pe fire", "intrări": len(OBS), "trecut": True})

cone_param_table = pd.DataFrame({f"{o}: {k}": v.astype(int) for o in OBS for k, v in cone_params[o].items()}, index=PARAM_NAMES)
display(CONE.T)

## 5. Configurația și directorul de rezultate

O rulare ulterioară cu altă configurație se oprește în loc să suprascrie. Exp. 9 și celelalte rezultate vechi sunt doar citite.

In [ ]:
def load_jsonl(path, required=False):
    if not path.exists():
        if required:
            raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def to_builtin(value):
    if isinstance(value, dict):
        return {k: to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_builtin(record), ensure_ascii=False) + "\n")


config = {
    "experiment": EXPERIMENT_NAME,
    "base_model": "Exp. 9 STAR ZZ (pairs (0,1),(0,2),(0,3), FEATURE_MAP_REPS=1, ANSATZ_REPS=3, 4 qubits, 32 params, 15 CNOT, 73 noise locations)",
    "change": "only the measured observable; score = (1 - <O>) / 2, no classical layer",
    "observables": {"Z0": "qml.PauliZ(0)", "Z1": "qml.PauliZ(1)", "Z2": "qml.PauliZ(2)", "Z3": "qml.PauliZ(3)", "Z0Z1": "qml.PauliZ(0) @ qml.PauliZ(1)"},
    "native_op_order": op_sequence(circuit_ops[BASELINE]),
    "baseline_weights": "results/result_exp9/weights/star (not retrained; verified)", "trained_from_scratch": NEW_OBS,
    "seeds": SEEDS, "noise_levels": P_ALL, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "optimizer": "Adam", "loss": "BCELoss", "init": "init_weights(seed), identical to Exp. 3 / 9, same for all observables",
    "shuffle": "torch.Generator().manual_seed(seed), identical to Exp. 3 / 9", "balancing": "undersample(X_train, y_train, seed), ratio 1:1",
    "encoder": "AngleEncoder(4, seed) fitted on the 1:1 set (Exp. 9), shared by all observables",
    "epoch_selection": "best validation AUPRC at p = 0",
    "noise_model": "noisy_qnode: NATIVE_GATES decomposition, DepolarizingChannel(p) after every gate on every wire, default.mixed, whole circuit",
    "threshold": "max F1 on validation scores at the same p (selected and evaluated on the same validation set: optimistic)",
    "recall_targets": RECALL_TARGETS, "recall_target_rule": "exploratory: threshold = score of the ceil(target * 99)-th highest-scored fraud; predicted fraud if score >= threshold",
    "auprc_tolerance_internal": AUPRC_TOLERANCE,
    "evaluation_set": "validation (test not used)", "split": "split_indices.npz (verified against official reconstruction)", "split_sha256": SPLIT_SHA256,
    "torch_default_dtype": str(TORCH_DEFAULT_DTYPE), "exact_tolerance": EXACT_TOL, "shots_per_execution": SHOTS_PER_EXECUTION,
    "cone_points": N_CONE_POINTS, "noise_cone_points": N_NOISE_CONE_POINTS, "smoke_rows": SMOKE_ROWS, "smoke_epochs": SMOKE_EPOCHS,
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__},
}
for d in [RESULTS_DIR, PLOTS_DIR, SCORES_DIR, *[WEIGHTS_DIR / o for o in NEW_OBS]]:
    d.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RESULTS_DIR / "config.json"
TRAIN_RUNS_PATH = RESULTS_DIR / "train_runs.jsonl"
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    current = json.loads(json.dumps(to_builtin(config)))
    if {k: v for k, v in previous.items() if k != "versions"} != {k: v for k, v in current.items() if k != "versions"}:
        raise RuntimeError(f"{CONFIG_PATH} conține altă configurație. Nu suprascriu: mută directorul sau schimbă RESULTS_DIR.")
else:
    CONFIG_PATH.write_text(json.dumps(to_builtin(config), indent=2, ensure_ascii=False), encoding="utf-8")
STRUCT.to_csv(RESULTS_DIR / "circuit_structure.csv")
CONE.to_csv(RESULTS_DIR / "causal_cone_analysis.csv")
cone_param_table.to_csv(RESULTS_DIR / "causal_cone_parameters.csv")
print("salvat: config.json, circuit_structure.csv, causal_cone_analysis.csv, causal_cone_parameters.csv")

## 6. Datele per seed, baseline-ul Z₀ (Exp. 9) și verificarea buclei de antrenare

Pentru fiecare seed: `undersample` 1:1 și `AngleEncoder(4, seed)`, calculate **o singură dată** și folosite de toate observabilele. Hash-ul unghiurilor (train + val) trebuie să fie identic cu cel salvat de Exp. 9 pentru STAR. Z₀ folosește ponderile STAR din Exp. 9 (AUPRC fără zgomot reprodus la 1e-12, inițializare = `init_weights(seed)`).

**Verificarea buclei:** reantrenăm STAR + Z₀ pentru seed 0 (rezultat nesalvat ca model) și cerem istoric, epocă aleasă și ponderi identice cu Exp. 9. Dacă trece, Z₁ … Z₀Z₁ sunt antrenate exact în condițiile Exp. 9, cu singura diferență observabila.

In [ ]:
exp9_train = {r["seed"]: r for r in load_jsonl(EXP9_DIR / "train_runs.jsonl", required=True) if r["topologie"] == "star"}
seed_data = {}
for seed in SEEDS:
    X_s, y_s = undersample(X_train, y_train, seed)
    assert len(y_s) == N_BALANCED and int(y_s.sum()) == N_BALANCED // 2 and X_s.index.is_unique and set(X_s.index) <= set(X_train.index)
    encoder = AngleEncoder(N_QUBITS, seed).fit(X_s)   # fitat doar pe cele 590 de exemple de train
    seed_data[seed] = {"Z_tr": encoder.transform(X_s), "y_tr": y_s, "Z_val": encoder.transform(X_val),
                       "balanced_index_sha256": hashlib.sha256(np.asarray(X_s.index).tobytes()).hexdigest()}
    seed_data[seed]["Z_sha256"] = hashlib.sha256(seed_data[seed]["Z_tr"].tobytes() + seed_data[seed]["Z_val"].tobytes()).hexdigest()
    assert seed_data[seed]["Z_sha256"] == exp9_train[seed]["sha256 unghiuri (train + val)"], f"seed {seed}: alte unghiuri decât Exp. 9"
checks.append({"verificare": "subseturi 1:1 și unghiuri (train + val) identice cu Exp. 9 (SHA-256), encoder fitat doar pe train", "intrări": len(SEEDS), "trecut": True})

models = {o: {} for o in OBS}
for seed in SEEDS:
    saved = torch.load(EXP9_DIR / "weights" / "star" / f"seed_{seed}.pt", weights_only=True)
    assert torch.equal(saved["initial_weights"], init_weights(seed).detach()), f"seed {seed}: inițializarea Exp. 9 ≠ init_weights(seed)"
    t0 = time.perf_counter()
    clean = predict(seed_data[seed]["Z_val"], saved["weights"], MODELS[BASELINE])
    elapsed = time.perf_counter() - t0
    ap = average_precision_score(y_val, clean)
    assert abs(ap - saved["best_val_ap"]) < EXACT_TOL, f"seed {seed}: {ap} vs {saved['best_val_ap']}"
    models[BASELINE][seed] = {"weights": saved["weights"], "best_epoch": saved["best_epoch"], "best_val_ap": saved["best_val_ap"],
                              "history": saved["history"], "clean": clean, "clean_time": elapsed,
                              "time_train_s": exp9_train[seed]["timp antrenare [s]"]}
    print(f"Z0 seed {seed}: ponderi STAR Exp. 9   epoca {saved['best_epoch']:2d}   val AUPRC (p = 0) {ap:.6f} = salvat {saved['best_val_ap']:.6f}")

s0 = SEEDS[0]
check_run = train_vqc_run(seed_data[s0]["Z_tr"], seed_data[s0]["y_tr"], seed_data[s0]["Z_val"], y_val, init_weights(s0), s0, MODELS[BASELINE])
hist_err = max(max(abs(a["train loss"] - b["train loss"]), abs(a["validation AUPRC"] - b["validation AUPRC"]))
               for a, b in zip(check_run["history"], models[BASELINE][s0]["history"]))
w_err = (check_run["weights"] - models[BASELINE][s0]["weights"]).abs().max().item()
assert check_run["best_epoch"] == models[BASELINE][s0]["best_epoch"] and hist_err < EXACT_TOL and w_err < EXACT_TOL, (hist_err, w_err)
assert check_run["batch_order_sha256"] == exp9_train[s0]["sha256 ordine mini-batch"]
Z0_RETRAIN_CHECK_TIME = check_run["time_train_s"]
checks.append({"verificare": "bucla de antrenare: STAR + Z0 seed 0 reantrenat = Exp. 9 (istoric, epocă, ponderi, ordine mini-batch)",
               "intrări": EPOCHS, "trecut": True, "eroare max": max(hist_err, w_err)})
print(f"✅ bucla reproduce Exp. 9 STAR (seed 0): eroare istoric {hist_err:.1e}, ponderi {w_err:.1e} ({Z0_RETRAIN_CHECK_TIME:.0f} s, nesalvat ca model)")

## 7. Test de fum (seed 0)

2 epoci pentru fiecare observabilă nouă (nesalvate) și o trecere cu zgomot p = 0.05 pe primele 2 000 de tranzacții din validation: rulează, `default.mixed` la p = 0 = `default.qubit`, scoruri în [0, 1].

In [ ]:
smoke_rows = []
for o in NEW_OBS:
    d = seed_data[s0]
    run = train_vqc_run(d["Z_tr"], d["y_tr"], d["Z_val"], y_val, init_weights(s0), s0, MODELS[o], epochs=SMOKE_EPOCHS)
    t0 = time.perf_counter()
    s_noisy = noisy_predict(d["Z_val"][:SMOKE_ROWS], run["weights"], P_FOCUS, CIRCUITS[o])
    noisy_time = time.perf_counter() - t0
    s_clean = predict(d["Z_val"][:SMOKE_ROWS], run["weights"], MODELS[o])
    sanity = np.abs(noisy_predict(d["Z_val"][:SMOKE_ROWS], run["weights"], 0.0, CIRCUITS[o]) - s_clean).max()
    assert sanity < 1e-10, f"{o}: default.mixed la p = 0 ≠ default.qubit ({sanity})"
    assert np.isfinite(s_noisy).all() and s_noisy.min() >= -EXACT_TOL and s_noisy.max() <= 1 + EXACT_TOL
    smoke_rows.append({"observabilă": o, f"timp antrenare {SMOKE_EPOCHS} epoci [s]": run["time_train_s"],
                       "val AUPRC epoca 2": run["history"][-1]["validation AUPRC"],
                       f"timp p = 0.05, {SMOKE_ROWS} tranz. [s]": noisy_time, "default.mixed p = 0 vs default.qubit": sanity})
    checks.append({"verificare": f"test de fum {o}: default.mixed p = 0 = default.qubit, scoruri în [0, 1]", "intrări": SMOKE_ROWS, "trecut": True, "eroare max": sanity})
smoke = pd.DataFrame(smoke_rows).set_index("observabilă")
display(smoke)
est_train = smoke[f"timp antrenare {SMOKE_EPOCHS} epoci [s]"].sum() / SMOKE_EPOCHS * EPOCHS * len(SEEDS) / 60
est_noise = smoke[f"timp p = 0.05, {SMOKE_ROWS} tranz. [s]"].mean() * len(y_val) / SMOKE_ROWS * len(NOISE_LEVELS) * len(SEEDS) * len(OBS) / 60
print(f"estimare: antrenare 4 observabile ~{est_train:.0f} min, evaluare cu zgomot (5 observabile) ~{est_noise:.0f} min")

## 8. Antrenarea Z₁, Z₂, Z₃, Z₀Z₁ (5 seeds, 30 de epoci, de la zero)

Aceeași buclă, aceeași inițializare `init_weights(seed)` și aceeași ordine a mini-batch-urilor ca STAR + Z₀ în Exp. 9 (hash SHA-256 al celor 30 de permutări comparat **per seed** cu Exp. 9). Fiecare model se salvează în `weights/<observabilă>/seed_<s>.pt` imediat după antrenare; o rulare întreruptă reia de unde a rămas.

In [ ]:
train_done = {(r["observabilă"], r["seed"]): r for r in load_jsonl(TRAIN_RUNS_PATH)}
for o in NEW_OBS:
    for seed in SEEDS:
        d = seed_data[seed]
        path = WEIGHTS_DIR / o / f"seed_{seed}.pt"
        if path.exists() and (o, seed) in train_done:
            saved = torch.load(path, weights_only=True)
            source = "încărcat"
        else:
            initial = init_weights(seed)
            run = train_vqc_run(d["Z_tr"], d["y_tr"], d["Z_val"], y_val, initial, seed, MODELS[o])
            saved = {"weights": run["weights"], "best_epoch": run["best_epoch"], "best_val_ap": run["best_val_ap"],
                     "history": run["history"], "initial_weights": initial.detach().clone()}
            torch.save(saved, path)
            losses = [h["train loss"] for h in run["history"]]
            append_jsonl(TRAIN_RUNS_PATH, {
                "observabilă": o, "seed": seed, "best_epoch": run["best_epoch"], "val AUPRC (p = 0)": run["best_val_ap"],
                "train loss la epoca aleasă": losses[run["best_epoch"] - 1], "train loss final": losses[-1],
                "epoci executate": len(run["history"]), "exemple train": len(d["y_tr"]), "timp antrenare [s]": run["time_train_s"],
                "sha256 ordine mini-batch": run["batch_order_sha256"], "sha256 unghiuri (train + val)": d["Z_sha256"]})
            train_done[(o, seed)] = load_jsonl(TRAIN_RUNS_PATH, required=True)[-1]
            source = f"antrenat și salvat ({run['time_train_s']:.0f} s)"
        record = train_done[(o, seed)]
        assert torch.equal(saved["initial_weights"], init_weights(seed).detach()), f"{o} seed {seed}: altă inițializare"
        assert record["sha256 unghiuri (train + val)"] == d["Z_sha256"], f"{o} seed {seed}: alte unghiuri"
        assert record["sha256 ordine mini-batch"] == exp9_train[seed]["sha256 ordine mini-batch"], f"{o} seed {seed}: altă ordine a mini-batch-urilor"
        assert len(saved["history"]) == EPOCHS and saved["best_epoch"] == 1 + int(np.argmax([h["validation AUPRC"] for h in saved["history"]]))
        t0 = time.perf_counter()
        clean = predict(d["Z_val"], saved["weights"], MODELS[o])
        elapsed = time.perf_counter() - t0
        assert abs(average_precision_score(y_val, clean) - saved["best_val_ap"]) < EXACT_TOL, f"{o} seed {seed}: ponderile nu reproduc AUPRC-ul salvat"
        models[o][seed] = {"weights": saved["weights"], "best_epoch": saved["best_epoch"], "best_val_ap": saved["best_val_ap"],
                           "history": saved["history"], "clean": clean, "clean_time": elapsed, "time_train_s": record["timp antrenare [s]"]}
        print(f"{o:<5} seed {seed}: {source}   epoca aleasă {saved['best_epoch']:2d}   val AUPRC (p = 0) {saved['best_val_ap']:.4f}"
              f"   (Z0 {models[BASELINE][seed]['best_val_ap']:.4f})")
checks.append({"verificare": "Z1, Z2, Z3, Z0Z1: 5 seeds × 30 epoci; aceeași inițializare, aceleași unghiuri, aceeași ordine a mini-batch-urilor ca Exp. 9",
               "intrări": len(NEW_OBS) * len(SEEDS), "trecut": True})
print("✅ toate modelele noi antrenate independent, ponderi salvate")

## 9. Evaluarea sub zgomot pe validation (toate cele cinci observabile)

Pentru fiecare (observabilă, seed, p): scorurile pe toată validarea (p = 0 cu `predict`, p > 0 cu `noisy_predict` oficial pe circuitul complet), salvate imediat în `scores/<o>_seed<s>_p<p>.npz` împreună cu timpul de simulare; o rulare întreruptă reia de unde a rămas. Testul nu este încărcat.

In [ ]:
def scores_path(o, seed, p):
    return SCORES_DIR / f"{o}_seed{seed}_p{p}.npz"


todo = [(o, seed, p) for o in OBS for seed in SEEDS for p in P_ALL if not scores_path(o, seed, p).exists()]
print(f"(observabilă, seed, p) de evaluat: {len(todo)} din {len(OBS) * len(SEEDS) * len(P_ALL)}")
for o, seed, p in todo:
    m = models[o][seed]
    if p == 0.0:
        scores, elapsed = m["clean"], m["clean_time"]
    else:
        t0 = time.perf_counter()
        scores = noisy_predict(seed_data[seed]["Z_val"], m["weights"], p, CIRCUITS[o])
        elapsed = time.perf_counter() - t0
    np.savez_compressed(scores_path(o, seed, p), val=scores, seconds=elapsed)
    print(f"{o:<5} seed {seed}  p = {p:<5}  AUPRC {average_precision_score(y_val, scores):.4f}  ({elapsed:.0f} s)")

SCORES, EVAL_TIME = {}, {}
for o in OBS:
    for seed in SEEDS:
        for p in P_ALL:
            z = np.load(scores_path(o, seed, p))
            SCORES[o, seed, p], EVAL_TIME[o, seed, p] = z["val"], float(z["seconds"])
            assert SCORES[o, seed, p].shape == y_val.shape and np.isfinite(SCORES[o, seed, p]).all()
            assert SCORES[o, seed, p].min() >= -EXACT_TOL and SCORES[o, seed, p].max() <= 1 + EXACT_TOL, (o, seed, p)
        assert np.array_equal(SCORES[o, seed, 0.0], models[o][seed]["clean"])
checks.append({"verificare": "scoruri complete (5 observabile × 5 seeds × 4 p), finite și în [0, 1]", "intrări": len(SCORES), "trecut": True})

## 10. Metrici per (observabilă, seed, p) și reproducerea STAR + Z₀

Prag max-F1 pe validation la același p (protocolul oficial; optimist pentru toate observabilele la fel). Spearman ρ, MAE, RMSE între scorurile cu și fără zgomot ale **aceluiași** model. `factor de contracție` = |⟨O⟩| mediu la p / |⟨O⟩| mediu la p = 0. Z₀ trebuie să reproducă exact (1e-12) metricile STAR din `result_exp9/val_runs.jsonl`.

In [ ]:
exp9_val = {(r["seed"], r["p"]): r for r in load_jsonl(EXP9_DIR / "val_runs.jsonl", required=True) if r["topologie"] == "star"}


def recall_target_point(y, s, target):
    # pragul = scorul celei de-a ceil(target * #fraude)-a fraude, în ordinea descrescătoare a scorurilor
    k = math.ceil(target * y.sum() - 1e-9)
    threshold = np.sort(s[y == 1])[::-1][k - 1]
    return threshold, evaluate(y, s, threshold)


rows, tradeoff_rows = [], []
for o in OBS:
    for seed in SEEDS:
        clean = SCORES[o, seed, 0.0]
        for p in P_ALL:
            s = SCORES[o, seed, p]
            threshold = choose_threshold(y_val, s)
            diff = s - clean
            rows.append({"observabilă": o, "seed": seed, "p": p, **evaluate(y_val, s, threshold), "threshold": float(threshold),
                         "Spearman rho": spearmanr(clean, s).statistic, "MAE scor": np.abs(diff).mean(),
                         "RMSE scor": float(np.sqrt(np.mean(diff ** 2))), "max |Δ scor|": np.abs(diff).max(),
                         "|⟨O⟩| mediu": float(np.mean(np.abs(1 - 2 * s))),
                         "factor de contracție |⟨O⟩|": float(np.mean(np.abs(1 - 2 * s)) / np.mean(np.abs(1 - 2 * clean))),
                         "timp evaluare [s]": EVAL_TIME[o, seed, p], "timp antrenare [s]": models[o][seed]["time_train_s"],
                         "epoca aleasă": models[o][seed]["best_epoch"], "CNOT": int(STRUCT.loc[o, "CNOT"]),
                         "locuri de zgomot": int(STRUCT.loc[o, "locuri de zgomot"]),
                         "locuri de zgomot în con": int(CONE.loc[o, "locuri de zgomot în con"]),
                         "parametri în conul pe fire": int(CONE.loc[o, "parametri în conul pe fire"]),
                         "parametri cu gradient nenul": int(CONE.loc[o, "parametri cu gradient nenul (exact)"])})
            for target in RECALL_TARGETS:
                thr, ev = recall_target_point(y_val, s, target)
                tradeoff_rows.append({"observabilă": o, "seed": seed, "p": p, "recall țintă": target, "prag": float(thr),
                                      "recall obținut": ev["recall"], "precision": ev["precision"], "F1": ev["F1"],
                                      "TP": ev["TP"], "FP": ev["FP"], "FN": ev["FN"]})
metrics = pd.DataFrame(rows)
a0 = metrics[metrics["p"] == 0.0].set_index(["observabilă", "seed"])["AUPRC"]
metrics["AUPRC_drop (vs p = 0)"] = a0.loc[list(zip(metrics["observabilă"], metrics["seed"]))].to_numpy() - metrics["AUPRC"]
metrics.to_csv(RESULTS_DIR / "metrics_per_seed.csv", index=False)
tradeoff = pd.DataFrame(tradeoff_rows)
tradeoff.to_csv(RESULTS_DIR / "recall_precision_tradeoff.csv", index=False)

repro_err = 0.0
for (seed, p), ref in exp9_val.items():
    r = metrics[(metrics["observabilă"] == BASELINE) & (metrics["seed"] == seed) & (metrics["p"] == p)].iloc[0]
    repro_err = max(repro_err, *(abs(r[k] - ref[k]) for k in ["AUPRC", "AUROC", "threshold", "F1", "MAE scor", "Spearman rho", "TP", "FP"]))
assert len(exp9_val) == len(SEEDS) * len(P_ALL) and repro_err < EXACT_TOL, repro_err
checks.append({"verificare": "STAR + Z0 reproduce Exp. 9 (AUPRC, AUROC, prag, F1, MAE, ρ, TP, FP) la toate seeds și p",
               "intrări": len(exp9_val), "trecut": True, "eroare max": repro_err})
print(f"✅ Z0 reproduce Exp. 9 STAR la {repro_err:.1e}; {len(metrics)} rânduri în metrics_per_seed.csv, {len(tradeoff)} în recall_precision_tradeoff.csv")

## 11. Referința clasică: regresie logistică pe aceleași unghiuri

Exact configurația Exp. 4 (`LogisticRegression(max_iter=1000)`, restul implicit) pe **aceleași patru unghiuri** STAR (encoderul Exp. 9 per seed, nu un PCA nou). Rezultatul trebuie să reproducă `result_exp4/val_runs.jsonl`. Modelul este clasic: aceleași scoruri la orice p.

In [ ]:
exp4_val = {r["seed"]: r for r in load_jsonl(EXP4_DIR / "val_runs.jsonl", required=True)}
lr_rows, LR_SCORES = [], {}
for seed in SEEDS:
    d = seed_data[seed]
    lr = LogisticRegression(max_iter=1000).fit(d["Z_tr"], d["y_tr"])
    s = lr.predict_proba(d["Z_val"])[:, 1]
    threshold = choose_threshold(y_val, s)
    LR_SCORES[seed] = s
    lr_rows.append({"seed": seed, **evaluate(y_val, s, threshold), "threshold": float(threshold)})
    assert abs(lr_rows[-1]["AUPRC"] - exp4_val[seed]["AUPRC"]) < EXACT_TOL and abs(threshold - exp4_val[seed]["threshold"]) < 1e-9
lr_metrics = pd.DataFrame(lr_rows)
checks.append({"verificare": "regresia logistică pe aceleași unghiuri reproduce Exp. 4 (AUPRC, prag)", "intrări": len(SEEDS), "trecut": True})
print(f"regresie logistică: AUPRC {lr_metrics['AUPRC'].mean():.4f} ± {lr_metrics['AUPRC'].std(ddof=1):.4f}   recall {lr_metrics['recall'].mean():.4f}   FP {lr_metrics['FP'].mean():.1f}")

## 12. Rezumat (medie ± std, ddof = 1, pe 5 seeds) și comparații pereche față de Z₀

Diferențele sunt **pereche**: același seed, aceleași unghiuri, aceeași inițializare și ordine a mini-batch-urilor; diferă doar observabila (și, în consecință, ponderile antrenate). Pentru fiecare metrică: Δ medie ± std și câte seeds favorizează observabila nouă. 5 seeds și 99 de fraude: fără semnificație statistică formală.

In [ ]:
METRICS = ["AUPRC", "AUROC", "Spearman rho", "precision", "recall", "F1", "MCC", "TP", "FP", "FN", "TN", "threshold",
           "MAE scor", "RMSE scor", "max |Δ scor|", "factor de contracție |⟨O⟩|", "AUPRC_drop (vs p = 0)", "timp evaluare [s]", "timp antrenare [s]"]
LOWER_IS_BETTER = {"MAE scor", "RMSE scor", "max |Δ scor|", "FP", "FN", "AUPRC_drop (vs p = 0)", "timp evaluare [s]", "timp antrenare [s]"}
NO_DIRECTION = {"threshold", "TN"}   # TN = negativi − FP, deja numărat prin FP
COUNTS = {"TP", "FP", "FN", "TN"}


def mean_std(values, digits=4):
    values = np.asarray(values, dtype=float)
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


def per_seed(o, p, metric):
    return metrics[(metrics["observabilă"] == o) & (metrics["p"] == p)].set_index("seed").loc[SEEDS, metric].astype(float)


summary_rows = []
for o in OBS:
    for p in P_ALL:
        row = {"observabilă": o, "p": p}
        for m in METRICS:
            v = per_seed(o, p, m)
            row[f"{m} medie"], row[f"{m} std"] = v.mean(), v.std(ddof=1)
        summary_rows.append(row)
for p in P_ALL:   # regresia logistică: aceleași scoruri la orice p
    row = {"observabilă": "LR (clasic, Exp. 4)", "p": p}
    for m in ["AUPRC", "AUROC", "precision", "recall", "F1", "MCC", "TP", "FP", "FN", "TN", "threshold"]:
        row[f"{m} medie"], row[f"{m} std"] = lr_metrics[m].mean(), lr_metrics[m].std(ddof=1)
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULTS_DIR / "summary_metrics.csv", index=False)

view = pd.DataFrame([{"observabilă": OBS_LABELS[o], "p": p, **{m: mean_std(per_seed(o, p, m), 1 if m in COUNTS else 4)
                      for m in ["AUPRC", "AUROC", "Spearman rho", "precision", "recall", "F1", "MCC", "TP", "FP"]}}
                     for o in OBS for p in P_ALL]).set_index(["observabilă", "p"])
print("Validation, medie ± std pe 5 seeds (prag max-F1 la același p):")
display(view)

PAIRED_METRICS = [m for m in METRICS if m not in NO_DIRECTION]
paired_rows, paired_seed_rows = [], []
for o in NEW_OBS:
    for p in P_ALL:
        for m in PAIRED_METRICS:
            base, new = per_seed(BASELINE, p, m), per_seed(o, p, m)
            d = new - base
            better = (d < 0) if m in LOWER_IS_BETTER else (d > 0)
            paired_rows.append({"comparație": f"{o} − Z0", "p": p, "metrică": m, "Z0 medie": base.mean(), "nou medie": new.mean(),
                                "Δ medie": d.mean(), "Δ std": d.std(ddof=1), "Δ min": d.min(), "Δ max": d.max(),
                                "seeds cu observabila nouă mai bună": int(better.sum()), "seeds egale": int((d == 0).sum())})
            for seed in SEEDS:
                paired_seed_rows.append({"comparație": f"{o} − Z0", "p": p, "metrică": m, "seed": seed, "Z0": base[seed], "nou": new[seed], "Δ": d[seed]})
paired = pd.DataFrame(paired_rows)
paired_seeds = pd.DataFrame(paired_seed_rows)
paired.to_csv(RESULTS_DIR / "paired_comparison.csv", index=False)
paired_seeds.to_csv(RESULTS_DIR / "paired_comparison_per_seed.csv", index=False)
print("Δ AUPRC pereche față de Z0, per seed:")
display(paired_seeds[paired_seeds["metrică"] == "AUPRC"].pivot_table(index=["comparație", "p"], columns="seed", values="Δ").round(4))

### Robustețe vs performanță de bază

`AUPRC_drop = AUPRC(p = 0) − AUPRC(p = 0.05)`, comparat pereche cu Z₀. O observabilă cu AUPRC mai mare la p = 0 **și** la p = 0.05, dar cu aceeași degradare, câștigă prin performanța de bază, nu prin robustețe.

In [ ]:
robust_rows = []
for o in OBS:
    a_0, a_5 = per_seed(o, 0.0, "AUPRC"), per_seed(o, P_FOCUS, "AUPRC")
    drop = a_0 - a_5
    row = {"observabilă": o, "AUPRC p = 0": mean_std(a_0), "AUPRC p = 0.05": mean_std(a_5), "AUPRC_drop": mean_std(drop),
           "AUPRC_drop relativ": mean_std(drop / a_0), "Spearman ρ p = 0.05": mean_std(per_seed(o, P_FOCUS, "Spearman rho"), 5),
           "MAE p = 0.05": mean_std(per_seed(o, P_FOCUS, "MAE scor")),
           "factor de contracție p = 0.05": mean_std(per_seed(o, P_FOCUS, "factor de contracție |⟨O⟩|")),
           "locuri de zgomot în con": int(CONE.loc[o, "locuri de zgomot în con"])}
    if o != BASELINE:
        base_0, base_5 = per_seed(BASELINE, 0.0, "AUPRC"), per_seed(BASELINE, P_FOCUS, "AUPRC")
        d_drop = drop - (base_0 - base_5)
        row.update({"Δ AUPRC p = 0 vs Z0": mean_std(a_0 - base_0), "Δ AUPRC p = 0.05 vs Z0": mean_std(a_5 - base_5),
                    "Δ AUPRC_drop vs Z0": mean_std(d_drop), "seeds cu degradare mai mică decât Z0": int((d_drop < 0).sum())})
    robust_rows.append(row)
robustness = pd.DataFrame(robust_rows).set_index("observabilă")
robustness.to_csv(RESULTS_DIR / "robustness_vs_baseline.csv")
display(robustness.T)

## 13. Alegerea observabilei (regulă fixată înainte de rulare)

La p = 0.05, pe validation, pentru fiecare observabilă nouă față de Z₀ (pereche pe seed):

1. **Câștig de calitate:** AUPRC mai mare decât Z₀ în medie **și** în ≥ 4 din 5 seeds → candidată.
2. **Câștig de recall în toleranță:** Δ AUPRC medie ≥ −0.02 (criteriu intern), recall mai mare la pragul oficial max-F1 în ≥ 4 din 5 seeds, iar Δ FP medie ≤ Δ TP medie (fiecare fraudă în plus costă cel mult o alarmă falsă în plus) → candidată.
3. Altfel, Z₀ rămâne. Printre candidate se alege AUPRC medie maximă la p = 0.05; Spearman ρ, stabilitatea între seeds și costul (Tracker) sunt raportate ca departajare și context.

In [ ]:
decision_rows = []
for o in NEW_OBS:
    g = lambda m: paired[(paired["comparație"] == f"{o} − Z0") & (paired["p"] == P_FOCUS) & (paired["metrică"] == m)].iloc[0]
    auprc, recall, tp, fp, rho = g("AUPRC"), g("recall"), g("TP"), g("FP"), g("Spearman rho")
    rule1 = auprc["Δ medie"] > 0 and auprc["seeds cu observabila nouă mai bună"] >= 4
    rule2 = auprc["Δ medie"] >= -AUPRC_TOLERANCE and recall["seeds cu observabila nouă mai bună"] >= 4 and fp["Δ medie"] <= tp["Δ medie"]
    decision_rows.append({"observabilă": o, "Δ AUPRC medie": auprc["Δ medie"], "seeds AUPRC mai bun": int(auprc["seeds cu observabila nouă mai bună"]),
                          "Δ recall medie": recall["Δ medie"], "seeds recall mai bun": int(recall["seeds cu observabila nouă mai bună"]),
                          "Δ TP medie": tp["Δ medie"], "Δ FP medie": fp["Δ medie"], "Δ Spearman ρ medie": rho["Δ medie"],
                          "AUPRC std între seeds": per_seed(o, P_FOCUS, "AUPRC").std(ddof=1),
                          "execuții / exemplu (Tracker)": CONE.loc[o, "execuții / exemplu (Tracker, parameter shift)"],
                          "regula 1 (calitate)": rule1, "regula 2 (recall în toleranță)": rule2, "candidată": rule1 or rule2})
decision = pd.DataFrame(decision_rows).set_index("observabilă")
candidates = decision[decision["candidată"]]
WINNER = BASELINE if candidates.empty else max(candidates.index, key=lambda o: per_seed(o, P_FOCUS, "AUPRC").mean())
decision.to_csv(RESULTS_DIR / "decision.csv")
display(decision)
print(f"Z0: AUPRC std între seeds la p = 0.05: {per_seed(BASELINE, P_FOCUS, 'AUPRC').std(ddof=1):.4f}, "
      f"execuții / exemplu (Tracker): {CONE.loc[BASELINE, 'execuții / exemplu (Tracker, parameter shift)']:.0f}")
print(f"➡ observabila păstrată conform regulii: {WINNER}")

### Sensibilitatea la numărul de shots (analiză suplimentară, adăugată post hoc)

Evaluarea oficială este exactă (`shots=None`, matrici de densitate). Pe hardware, ⟨O⟩ se estimează din N shots. Pentru toate cele cinci observabile (diagonale în baza computațională), scorul exact s = (1 − ⟨O⟩)/2 este **exact probabilitatea rezultatului −1** al lui O, deci estimatorul cu N shots este ŝ = Binomial(N, s) / N, independent pentru fiecare tranzacție. Îl eșantionăm pe scorurile exacte salvate (care includ deja zgomotul depolarizant), cu N = 1024 (bugetul din cost ledger), 10 000 și 100 000, 10 repetări per (observabilă, seed, p), `default_rng(42)`. Fără erori de readout, fără alt model de zgomot.

**Statut:** această analiză a fost adăugată **după** ce am văzut că la p = 0.05 scorurile tuturor modelelor sunt comprimate spre 0.5 (factor de contracție 0.05–0.11). Regula de alegere din §13 rămâne cea fixată înainte; rezultatele de aici se raportează separat, ca informație despre costul pe hardware.

In [ ]:
SHOT_LEVELS = [1024, 10_000, 100_000]
N_SHOT_REPS = 10
rng_shots = np.random.default_rng(SPLIT_SEED)
shot_rows = []
for o in OBS:
    for seed in SEEDS:
        for p in P_ALL:
            s = np.clip(SCORES[o, seed, p], 0, 1)
            gap = s[y_val == 1].mean() - s[y_val == 0].mean()
            for n_shots in SHOT_LEVELS:
                reps = []
                for _ in range(N_SHOT_REPS):
                    s_hat = rng_shots.binomial(n_shots, s) / n_shots
                    reps.append({"AUPRC": average_precision_score(y_val, s_hat),
                                 **{k: v for k, v in evaluate(y_val, s_hat, choose_threshold(y_val, s_hat)).items() if k in ("recall", "TP", "FP", "F1")}})
                reps = pd.DataFrame(reps)
                shot_rows.append({"observabilă": o, "seed": seed, "p": p, "shots": n_shots, "AUPRC exact": average_precision_score(y_val, s),
                                  "AUPRC shots (medie repetări)": reps["AUPRC"].mean(), "AUPRC shots (std repetări)": reps["AUPRC"].std(ddof=1),
                                  "recall shots": reps["recall"].mean(), "TP shots": reps["TP"].mean(), "FP shots": reps["FP"].mean(),
                                  "F1 shots": reps["F1"].mean(), "diferența medie de scor fraudă − legitim": gap,
                                  "std scor": s.std(), "eroare standard shot la s = 0.5": 0.5 / np.sqrt(n_shots)})
shots_df = pd.DataFrame(shot_rows)
shots_df.to_csv(RESULTS_DIR / "shot_noise_sensitivity.csv", index=False)
shot_summary = (shots_df.groupby(["p", "shots", "observabilă"])[["AUPRC exact", "AUPRC shots (medie repetări)", "recall shots", "FP shots", "diferența medie de scor fraudă − legitim"]]
                .agg(["mean", "std"]))
display(shot_summary.loc[P_FOCUS].round(4))

shot_paired = []
for o in NEW_OBS:
    for p in P_ALL:
        for n_shots in SHOT_LEVELS:
            sel = lambda obs: shots_df[(shots_df["observabilă"] == obs) & (shots_df["p"] == p) & (shots_df["shots"] == n_shots)].set_index("seed").loc[SEEDS, "AUPRC shots (medie repetări)"]
            d = sel(o) - sel(BASELINE)
            shot_paired.append({"comparație": f"{o} − Z0", "p": p, "shots": n_shots, "Δ AUPRC medie": d.mean(), "Δ AUPRC std": d.std(ddof=1),
                                "seeds cu observabila nouă mai bună": int((d > 0).sum())})
shot_paired = pd.DataFrame(shot_paired)
shot_paired.to_csv(RESULTS_DIR / "shot_noise_paired_vs_z0.csv", index=False)
display(shot_paired[shot_paired["p"] == P_FOCUS].set_index(["comparație", "shots"]).drop(columns="p").round(4))

## 14. Cost ledger

Patru mărimi separate, ca în Exp. 9:
1. **Structura per execuție** (porți, CNOT, adâncime, locuri de zgomot): identică pentru toate observabilele (doar măsurarea diferă).
2. **Execuții de circuit**: antrenare = epoci × 590 × execuții / exemplu, nominal (1 + 2P, P = 32) și **măsurat cu `qml.Tracker`** (parameter shift pe hardware; simulatorul folosește backprop, deci nu le-a executat); validare după fiecare epocă (p = 0); trecerea pe validation la p = 0 o dată + câte una per p cu zgomot.
3. **Shots nominale** = execuții × 1024. Nu au fost trase (simulare exactă). Pentru Z₀Z₁, ambele rezultate Z vin din aceeași măsurare în baza computațională: aceleași shots ca pentru Z₀, nu dublu.
4. **Timpul simulatorului**, informativ.

Pași Adam per seed = 30 × ⌈590 / 32⌉ = 570.

In [ ]:
n_val = len(y_val)
batches_per_epoch = math.ceil(N_BALANCED / BATCH_SIZE)
N_PARAMS = 32
ledger = {"shots_per_execution": SHOTS_PER_EXECUTION, "seeds": len(SEEDS), "adam_steps_per_seed": EPOCHS * batches_per_epoch,
          "convention": "hardware estimate with parameter shift; the simulator trained with backprop (exact, shots=None), so the shot counts were NOT drawn",
          "observables": {}}
ledger_rows = []
for o in OBS:
    tracker = float(CONE.loc[o, "execuții / exemplu (Tracker, parameter shift)"])
    comp = {
        "train_forward_plus_parameter_shift_nominal_1+2P": EPOCHS * N_BALANCED * (1 + 2 * N_PARAMS),
        "train_forward_plus_parameter_shift_tracker": EPOCHS * N_BALANCED * tracker,
        "validation_after_each_epoch_p0": EPOCHS * n_val,
        "validation_p0_selected_checkpoint": n_val,
        "validation_noisy_p0.001_0.01_0.05": len(NOISE_LEVELS) * n_val,
    }
    total_nominal = sum(v for k, v in comp.items() if not k.endswith("tracker"))
    total_tracker = sum(v for k, v in comp.items() if not k.endswith("1+2P"))
    t_train = per_seed(o, 0.0, "timp antrenare [s]")
    t_noisy = sum(per_seed(o, p, "timp evaluare [s]") for p in NOISE_LEVELS)
    ledger["observables"][o] = {
        "status": "reused from Exp. 9 (training not repeated; noisy validation recomputed and verified)" if o == BASELINE else "trained and evaluated in Exp. 11",
        "nominal_parameters": N_PARAMS, "parameters_in_wire_cone": int(CONE.loc[o, "parametri în conul pe fire"]),
        "parameters_nonzero_gradient": int(CONE.loc[o, "parametri cu gradient nenul (exact)"]),
        "executions_per_example_nominal": 1 + 2 * N_PARAMS, "executions_per_example_tracker": tracker,
        "executions_per_example_if_only_nonzero_gradient_shifted_NOT_IMPLEMENTED": 1 + 2 * int(CONE.loc[o, "parametri cu gradient nenul (exact)"]),
        "cnot_per_execution": int(STRUCT.loc[o, "CNOT"]), "noise_locations_per_execution": int(STRUCT.loc[o, "locuri de zgomot"]),
        "noise_locations_in_cone": int(CONE.loc[o, "locuri de zgomot în con"]), "basis_rotations": 0,
        "executions_per_seed": comp, "total_per_seed_nominal": total_nominal, "total_per_seed_tracker": total_tracker,
        "total_5_seeds_nominal": total_nominal * len(SEEDS), "total_5_seeds_tracker": total_tracker * len(SEEDS),
        "shots_5_seeds_nominal": total_nominal * len(SEEDS) * SHOTS_PER_EXECUTION,
        "shots_5_seeds_tracker": total_tracker * len(SEEDS) * SHOTS_PER_EXECUTION,
        "simulator_train_seconds_per_seed_mean": float(t_train.mean()), "simulator_noisy_eval_seconds_per_seed_mean": float(t_noisy.mean()),
    }
    for k, v in comp.items():
        ledger_rows.append({"observabilă": o, "componentă": k, "execuții / seed": v, "execuții 5 seeds": v * len(SEEDS),
                            "shots nominale 5 seeds": v * len(SEEDS) * SHOTS_PER_EXECUTION})
    for k, v in [("TOTAL nominal", total_nominal), ("TOTAL Tracker", total_tracker)]:
        ledger_rows.append({"observabilă": o, "componentă": k, "execuții / seed": v, "execuții 5 seeds": v * len(SEEDS),
                            "shots nominale 5 seeds": v * len(SEEDS) * SHOTS_PER_EXECUTION})
base_tracker_total = ledger["observables"][BASELINE]["total_per_seed_tracker"]
for o in OBS:
    ledger["observables"][o]["tracker_total_vs_Z0_relative"] = ledger["observables"][o]["total_per_seed_tracker"] / base_tracker_total - 1
ledger["experiment_overhead"] = {
    "new_trainings": f"{len(NEW_OBS)} observables x {len(SEEDS)} seeds (Z0 reused from Exp. 9)",
    "new_training_executions_tracker": sum(EPOCHS * N_BALANCED * float(CONE.loc[o, "execuții / exemplu (Tracker, parameter shift)"]) for o in NEW_OBS) * len(SEEDS),
    "new_validation_executions_during_training": len(NEW_OBS) * len(SEEDS) * (EPOCHS + 1) * n_val,
    "noisy_validation_executions_all_observables": len(OBS) * len(SEEDS) * len(NOISE_LEVELS) * n_val,
    "z0_loop_check_retrain_seed0_simulator_seconds": Z0_RETRAIN_CHECK_TIME,
    "simulator_minutes_new_trainings": sum(per_seed(o, 0.0, "timp antrenare [s]").sum() for o in NEW_OBS) / 60,
    "simulator_minutes_noisy_evaluation": sum(per_seed(o, p, "timp evaluare [s]").sum() for o in OBS for p in NOISE_LEVELS) / 60,
}
(RESULTS_DIR / "cost_ledger.json").write_text(json.dumps(to_builtin(ledger), indent=2, ensure_ascii=False), encoding="utf-8")
cost_table = pd.DataFrame(ledger_rows).set_index(["observabilă", "componentă"])
cost_table.to_csv(RESULTS_DIR / "cost_ledger.csv")
display(pd.DataFrame({o: {k: v for k, v in ledger["observables"][o].items() if not isinstance(v, dict)} for o in OBS}))
print(json.dumps(to_builtin(ledger["experiment_overhead"]), indent=2))

## 15. Grafice

In [ ]:
xpos = np.arange(len(P_ALL))
xlabels_p = ["0", *[str(p) for p in NOISE_LEVELS]]
OFFSETS = dict(zip(OBS, np.linspace(-0.16, 0.16, len(OBS))))
LR_MEAN = {m: lr_metrics[m].mean() for m in ["AUPRC", "recall", "F1"]}


def plot_vs_p(metric, ylabel, title, filename, legend_loc="best", lr=True):
    fig, ax = plt.subplots(figsize=(8, 4.6))
    for o in OBS:
        values = [per_seed(o, p, metric) for p in P_ALL]
        for i, v in enumerate(values):
            ax.scatter(np.full(len(v), xpos[i] + OFFSETS[o]), v, s=10, color=SEED_DOT_COLOR, alpha=0.55, zorder=2)
        ax.errorbar(xpos + OFFSETS[o], [v.mean() for v in values], yerr=[v.std(ddof=1) for v in values], color=OBS_COLORS[o],
                    marker=OBS_MARKERS[o], markersize=8, linewidth=2, capsize=3, zorder=3, label=OBS_LABELS[o])
    if lr and metric in LR_MEAN:
        ax.axhline(LR_MEAN[metric], color=LR_COLOR, linestyle=":", linewidth=1.5, label=f"regresie logistică (clasic) {LR_MEAN[metric]:.3f}")
    ax.set_xticks(xpos, xlabels_p)
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8.5, loc=legend_loc, ncol=2)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=130)
    plt.show()


plot_vs_p("AUPRC", "AUPRC (validation)", "1 · AUPRC vs p · medie ± std pe 5 seeds (gri: seeds)", "1_auprc_vs_p.png", "lower left")
plot_vs_p("recall", "recall la pragul max-F1 (același p)", "2 · Recall vs p · medie ± std · prag max-F1 pe validation (optimist)", "2_recall_vs_p.png", "lower left")
plot_vs_p("F1", "F1 la pragul max-F1 (același p)", "3 · F1 vs p · medie ± std · prag max-F1 pe validation (optimist)", "3_f1_vs_p.png", "lower left")
plot_vs_p("Spearman rho", "Spearman ρ (scoruri cu vs fără zgomot)", "4 · Spearman ρ vs p · medie ± std pe 5 seeds", "4_spearman_vs_p.png", "lower left", lr=False)

In [ ]:
def per_seed_bars(metric, ylabel, title, filename, digits=3):
    fig, ax = plt.subplots(figsize=(9.5, 4.6))
    width = 0.8 / len(OBS)
    for k, o in enumerate(OBS):
        v = per_seed(o, P_FOCUS, metric)
        ax.bar(np.arange(len(SEEDS)) + (k - (len(OBS) - 1) / 2) * width, v.to_numpy(), width=width - 0.02, color=OBS_COLORS[o],
               label=f"{OBS_LABELS[o]} (medie {v.mean():.{digits}f})")
    if metric in LR_MEAN:
        ax.axhline(LR_MEAN[metric], color=LR_COLOR, linestyle=":", linewidth=1.5, label=f"regresie logistică (medie {LR_MEAN[metric]:.3f})")
    ax.set_xticks(range(len(SEEDS)), [f"seed {s}" for s in SEEDS])
    ax.set_ylabel(ylabel)
    low = min(per_seed(o, P_FOCUS, metric).min() for o in OBS)
    ax.set_ylim(max(0, low - 0.1), None)
    ax.set_title(title)
    ax.legend(fontsize=8, loc="lower right", framealpha=0.95, ncol=2)
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=130)
    plt.show()


per_seed_bars("AUPRC", "AUPRC la p = 0.05 (validation)", "5 · AUPRC per seed la p = 0.05", "5_auprc_per_seed_p0.05.png")
per_seed_bars("recall", "recall la p = 0.05 (prag max-F1)", "6 · Recall per seed la p = 0.05 (prag max-F1, optimist)", "6_recall_per_seed_p0.05.png")

# 7. curbe precision-recall la p = 0.05: un panou per observabilă, seeds subțiri; media Z0 (interpolată) ca referință
recall_grid = np.linspace(0, 1, 201)


def interp_precision(y, s):
    precision, recall, _ = precision_recall_curve(y, s)   # recall descrescător
    return np.interp(recall_grid, recall[::-1], precision[::-1])


z0_mean_pr = np.mean([interp_precision(y_val, SCORES[BASELINE, s, P_FOCUS]) for s in SEEDS], axis=0)
fig, axes = plt.subplots(1, len(OBS), figsize=(18, 4.2), sharey=True)
for ax, o in zip(axes, OBS):
    for seed in SEEDS:
        precision, recall, _ = precision_recall_curve(y_val, SCORES[o, seed, P_FOCUS])
        ax.plot(recall, precision, color=OBS_COLORS[o], linewidth=1, alpha=0.6)
    if o != BASELINE:
        ax.plot(recall_grid, z0_mean_pr, color=SEED_DOT_COLOR, linestyle="--", linewidth=1.5, label="Z₀: media pe seeds")
        ax.legend(fontsize=8, loc="lower left")
    ax.set_title(f"{OBS_LABELS[o]} · AUPRC {per_seed(o, P_FOCUS, 'AUPRC').mean():.3f}", fontsize=10)
    ax.set_xlabel("recall")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
axes[0].set_ylabel("precision")
fig.suptitle("7 · Curbe precision-recall la p = 0.05 (validation; câte o linie per seed)", fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "7_pr_curves_p0.05.png", dpi=130)
plt.show()

# 8. recall vs false positives la p = 0.05 (bugetul de alarme false), medie ± std pe seeds; ★ = punctul max-F1
MAX_FP = 200


def recall_at_fp(y, s, max_fp=MAX_FP):
    order = np.argsort(-s, kind="mergesort")
    tp, fp = np.cumsum(y[order]), np.cumsum(1 - y[order])
    idx = np.searchsorted(fp, np.arange(max_fp + 1), side="right") - 1
    return np.where(idx >= 0, tp[np.clip(idx, 0, None)], 0) / y.sum()


fig, ax = plt.subplots(figsize=(8.5, 5))
budget = np.arange(MAX_FP + 1)
for o in OBS:
    curves = np.array([recall_at_fp(y_val, SCORES[o, s, P_FOCUS]) for s in SEEDS])
    mean, std = curves.mean(0), curves.std(0, ddof=1)
    ax.fill_between(budget, mean - std, mean + std, color=OBS_COLORS[o], alpha=0.12, linewidth=0)
    ax.plot(budget, mean, color=OBS_COLORS[o], linewidth=2, label=OBS_LABELS[o])
    ax.scatter(per_seed(o, P_FOCUS, "FP").mean(), per_seed(o, P_FOCUS, "recall").mean(), s=120, marker="*", color=OBS_COLORS[o],
               edgecolor="white", linewidth=1, zorder=4)
for target in RECALL_TARGETS:
    ax.axhline(target, color="0.6", linewidth=0.8, linestyle=":")
    ax.annotate(f"{target:.0%}", (MAX_FP, target), textcoords="offset points", xytext=(4, 0), va="center", fontsize=8, color="0.4")
ax.set_ylim(0, 0.9)
ax.set_xlabel("false positives permise (din 56 863 tranzacții legitime în validation)")
ax.set_ylabel("recall (fraude detectate / 99)")
ax.set_title("8 · Recall vs false positives la p = 0.05 · medie ± std pe 5 seeds · ★ = pragul max-F1")
ax.legend(fontsize=9, loc="lower right")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "8_recall_vs_fp_p0.05.png", dpi=130)
plt.show()

In [ ]:
# 9. Δ AUPRC pereche față de Z0: per seed (gri), medie (culoare), la fiecare p
fig, axes = plt.subplots(1, len(NEW_OBS), figsize=(16, 4.2), sharey=True)
for ax, o in zip(axes, NEW_OBS):
    for seed in SEEDS:
        d = [per_seed(o, p, "AUPRC")[seed] - per_seed(BASELINE, p, "AUPRC")[seed] for p in P_ALL]
        ax.plot(xpos, d, color=SEED_DOT_COLOR, marker="o", markersize=4, linewidth=1, alpha=0.8, zorder=2)
        ax.annotate(f"s{seed}", (xpos[-1], d[-1]), textcoords="offset points", xytext=(5, 0), fontsize=7.5, color="0.35", va="center")
    mean_d = [(per_seed(o, p, "AUPRC") - per_seed(BASELINE, p, "AUPRC")).mean() for p in P_ALL]
    ax.plot(xpos, mean_d, color=OBS_COLORS[o], marker=OBS_MARKERS[o], markersize=8, linewidth=2.5, label="medie pe 5 seeds", zorder=3)
    ax.axhline(0, color="0.3", linestyle="--", linewidth=1)
    ax.axhspan(-AUPRC_TOLERANCE, 0, color="0.85", alpha=0.4, zorder=0, label="toleranța internă −0.02")
    ax.set_xticks(xpos, xlabels_p)
    ax.set_xlabel("p")
    ax.set_title(f"Δ AUPRC ({OBS_LABELS[o]} − Z₀)", fontsize=10)
    ax.legend(loc="lower left", fontsize=8)
axes[0].set_ylabel("Δ AUPRC (> 0: observabila nouă e mai bună)")
fig.suptitle("9 · Diferențe pereche pe seed față de Z₀ · validation", fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "9_paired_delta_auprc_vs_z0.png", dpi=130)
plt.show()

# 10a. conul cauzal pe porți: grilă poartă × qubit, porți în con colorate, canale de zgomot din con marcate
fig, axes = plt.subplots(len(OBS), 1, figsize=(15, 2.0 * len(OBS)), sharex=True)
ops = circuit_ops[BASELINE]
for ax, o in zip(axes, OBS):
    in_cone, noisy_wires = cone_masks[o]["in_cone"], cone_masks[o]["noisy_wires"]
    for k, op in enumerate(ops):
        wires = list(op.wires)
        color = OBS_COLORS[o] if in_cone[k] else "#e3e4e6"
        if len(wires) == 2:
            ax.plot([k, k], wires, color=color, linewidth=1.5, zorder=1)
        for w in wires:
            ax.scatter(k, w, s=55 if op.name != "CNOT" or w == wires[0] else 75, marker="o" if op.name != "CNOT" or w == wires[0] else "P",
                       color=color, edgecolor="white", linewidth=0.8, zorder=2)
    ax.axvline(N_FM_OPS - 0.5, color="0.4", linestyle="--", linewidth=1)
    for w in OBSERVABLE_WIRES[o]:
        ax.annotate("⟨Z⟩", (len(ops) - 0.2, w), fontsize=9, va="center", color="0.15", fontweight="bold")
    ax.set_yticks(range(N_QUBITS), [f"q{q}" for q in range(N_QUBITS)])
    ax.set_ylim(N_QUBITS - 0.5, -0.5)
    ax.grid(False)
    ax.set_title(f"{OBS_LABELS[o]}: {int(CONE.loc[o, 'porți în con (pe fire)'])}/58 porți, {int(CONE.loc[o, 'locuri de zgomot în con'])}/73 locuri de zgomot, "
                 f"{int(CONE.loc[o, 'parametri în conul pe fire'])} parametri în con, {int(CONE.loc[o, 'parametri cu gradient nenul (exact)'])} cu gradient nenul",
                 fontsize=9.5, loc="left")
axes[-1].set_xlabel("indexul porții native în circuit (stânga liniei punctate: feature map STAR; dreapta: ansatz)")
fig.suptitle("10a · Conul cauzal invers al fiecărei observabile (gri: porți care nu pot influența măsurarea)", fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "10a_causal_cone_gates.png", dpi=120)
plt.show()

# 10b. parametrii ansatz-ului: în afara conului / în con dar cu gradient structural 0 / gradient nenul
fig, ax = plt.subplots(figsize=(15, 3.2))
level = np.array([[0 if not cone_params[o]["con pe fire"][i] else (1 if not cone_params[o]["gradient nenul"][i] else 2) for i in range(32)] for o in OBS])
cmap = ListedColormap(["#eceef1", "#86b6ef", "#1c5cab"])
ax.imshow(level, cmap=cmap, vmin=0, vmax=2, aspect="auto")
ax.set_yticks(range(len(OBS)), [f"{OBS_LABELS[o]}  ({int(CONE.loc[o, 'execuții / exemplu (Tracker, parameter shift)'])} exec./exemplu)" for o in OBS])
ax.set_xticks(range(32), PARAM_NAMES, rotation=90, fontsize=7)
for l in range(1, ANSATZ_REPS + 1):
    ax.axvline(8 * l - 0.5, color="white", linewidth=2)
ax.grid(False)
handles = [Patch(color=c, label=t) for c, t in zip(cmap.colors, ["în afara conului (nu e deplasat de parameter shift)",
           "în conul pe fire, gradient structural 0 (deplasat inutil)", "gradient nenul"])]
ax.legend(handles=handles, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.55), ncol=3)
ax.set_title("10b · Parametrii care pot influența fiecare observabilă (θ[strat, qubit, poartă])", fontsize=11)
fig.savefig(PLOTS_DIR / "10b_causal_cone_parameters.png", dpi=130, bbox_inches="tight")
plt.show()

# 11. AUPRC la p = 0.05 vs numărul de shots (analiză post hoc)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
shot_x = np.arange(len(SHOT_LEVELS) + 1)
for k, o in enumerate(OBS):
    sel = shots_df[(shots_df["observabilă"] == o) & (shots_df["p"] == P_FOCUS)]
    per_n = [sel[sel["shots"] == n].set_index("seed").loc[SEEDS, "AUPRC shots (medie repetări)"] for n in SHOT_LEVELS] + [per_seed(o, P_FOCUS, "AUPRC")]
    dx = (k - (len(OBS) - 1) / 2) * 0.06
    for i, v in enumerate(per_n):
        ax.scatter(np.full(len(v), shot_x[i] + dx), v, s=10, color=SEED_DOT_COLOR, alpha=0.55, zorder=2)
    ax.errorbar(shot_x + dx, [v.mean() for v in per_n], yerr=[v.std(ddof=1) for v in per_n], color=OBS_COLORS[o], marker=OBS_MARKERS[o],
                markersize=8, linewidth=2, capsize=3, zorder=3, label=OBS_LABELS[o])
ax.axhline(LR_MEAN["AUPRC"], color=LR_COLOR, linestyle=":", linewidth=1.5, label=f"regresie logistică (clasic) {LR_MEAN['AUPRC']:.3f}")
ax.set_xticks(shot_x, [f"{n:,}".replace(",", " ") for n in SHOT_LEVELS] + ["exact\n(protocol oficial)"])
ax.set_xlabel("shots per execuție")
ax.set_ylabel("AUPRC la p = 0.05 (validation)")
ax.set_title("11 · AUPRC la p = 0.05 cu zgomot de eșantionare (post hoc) · medie ± std pe 5 seeds")
ax.legend(fontsize=8.5, loc="lower right", ncol=2)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "11_auprc_vs_shots_p0.05.png", dpi=130)
plt.show()

pd.DataFrame(checks).to_csv(RESULTS_DIR / "checks.csv", index=False)
print(f"grafice salvate în {PLOTS_DIR.relative_to(REPO_ROOT)}; {len(checks)} verificări în checks.csv")

## 16. Cum citim rezultatele

- **Rezultat principal:** pragul oficial max-F1 la același p (precision, recall, F1, MCC, matricea de confuzie). Țintele de recall 70–85% (`recall_precision_tradeoff.csv`, graficul 8) sunt **exploratorii**: mutarea pragului nu schimbă AUPRC și nu este o îmbunătățire cuantică sau anti-zgomot.
- **Performanță vs robustețe:** AUPRC la p = 0 măsoară clasificatorul; AUPRC_drop, Spearman ρ, MAE și factorul de contracție măsoară cât îl strică zgomotul. Cele două se raportează separat.
- **Conul cauzal** explică ce porți, parametri și canale de zgomot pot influența fiecare observabilă; zgomotul a fost totuși aplicat întregului circuit în toate evaluările.
- **Costul:** porțile, CNOT și locurile de zgomot per execuție sunt identice; diferă doar câți parametri deplasează parameter shift (măsurat cu `qml.Tracker`). Shots pentru Z₀Z₁ = aceleași ca pentru Z₀ (aceeași bază de măsurare).
- **Shots (post hoc):** evaluarea oficială este exactă. O observabilă cu semnal mai comprimat (diferență de scor fraudă − legitim mai mică) are nevoie de mai multe shots pentru același clasament; `shot_noise_sensitivity.csv` și graficul 11.
- Diferențele sunt pereche; 5 seeds și 99 de fraude nu dau semnificație statistică formală. Testul nu a fost folosit. Interpretarea completă: `results/result_exp11/report.md`.